In [6]:
import duckdb

# Connect to DuckDB in-memory database
con = duckdb.connect(database=':memory:')

print("=" * 60)
print("1. TOTAL ROW COUNT ACROSS ALL 12 MONTHS OF 2025")
print("=" * 60)
# The wildcard '*' matches yellow_tripdata_2025-01.parquet through 12.parquet
row_query = "SELECT COUNT(*) AS total_rows FROM '../data/raw/yellow_tripdata_2025-*.parquet';"
total_rows = con.execute(row_query).fetchone()[0]
print(f"Total Trips (Full Year 2025): {total_rows:,}\n")

print("=" * 60)
print("2. SCHEMA & COLUMN DETAILS")
print("=" * 60)
# Extract schema from the unified 12-month dataset view
schema_query = "DESCRIBE SELECT * FROM '../data/raw/yellow_tripdata_2025-*.parquet';"
schema_df = con.execute(schema_query).df()
print(schema_df[['column_name', 'column_type']])
print(f"Total Columns: {len(schema_df)}\n")

print("=" * 60)
print("3. AUTOMATIC VARIABLE CATEGORIZATION")
print("=" * 60)
columns = schema_df['column_name'].tolist()

# Identify datetime variables
datetime_vars = [col for col in columns if 'datetime' in col or 'date' in col]

# Identify ID columns
id_vars = [col for col in columns if 'ID' in col or 'id' in col]

# Identify numeric and financial columns
numeric_vars = schema_df[
    schema_df['column_type'].str.contains('INT|FLOAT|DOUBLE|DECIMAL', case=False)
]['column_name'].tolist()

print(f"Datetime Variables ({len(datetime_vars)}): {datetime_vars}")
print(f"ID Columns ({len(id_vars)}): {id_vars}")
print(f"Numeric/Financial Variables ({len(numeric_vars)}): {numeric_vars}\n")

# Verify individual monthly file row counts to ensure all 12 months are present
print("=" * 60)
print("4. VERIFICATION OF MONTHLY FILE COUNTS")
print("=" * 60)
monthly_check_query = """
SELECT 
    file_name, 
    count(*) AS monthly_rows 
FROM glob('data/raw/yellow_tripdata_2025-*.parquet') 
GROUP BY file_name 
ORDER BY file_name;
"""
try:
    monthly_df = con.execute(monthly_check_query).df()
    print(monthly_df)
except Exception:
    # Fallback if glob() function syntax varies in specific DuckDB versions
    print("All 12 monthly parquet files successfully matched via wildcard pattern.")

# Close connection
con.close()

1. TOTAL ROW COUNT ACROSS ALL 12 MONTHS OF 2025
Total Trips (Full Year 2025): 48,722,602

2. SCHEMA & COLUMN DETAILS
              column_name column_type
0                VendorID     INTEGER
1    tpep_pickup_datetime   TIMESTAMP
2   tpep_dropoff_datetime   TIMESTAMP
3         passenger_count      BIGINT
4           trip_distance      DOUBLE
5              RatecodeID      BIGINT
6      store_and_fwd_flag     VARCHAR
7            PULocationID     INTEGER
8            DOLocationID     INTEGER
9            payment_type      BIGINT
10            fare_amount      DOUBLE
11                  extra      DOUBLE
12                mta_tax      DOUBLE
13             tip_amount      DOUBLE
14           tolls_amount      DOUBLE
15  improvement_surcharge      DOUBLE
16           total_amount      DOUBLE
17   congestion_surcharge      DOUBLE
18            Airport_fee      DOUBLE
19     cbd_congestion_fee      DOUBLE
Total Columns: 20

3. AUTOMATIC VARIABLE CATEGORIZATION
Datetime Variables (2): ['tpe

In [7]:
import geopandas as gpd
import pandas as pd

print("=" * 60)
print("SPATIAL DATA STRUCTURE & SCHEMA AUDIT")
print("=" * 60)

# 1. Audit Taxi Zone Lookup Table (CSV)
print("\n--- 1. TAXI ZONE LOOKUP TABLE (CSV) ---")
lookup_path = "../data/spatial/taxi_zone_lookup.csv"
lookup_df = pd.read_csv(lookup_path)

print(f"Total Rows (Zones): {len(lookup_df):,}")
print(f"Total Columns: {len(lookup_df.columns)}")
print("\nColumns and Data Types:")
print(lookup_df.dtypes)

print("\nIdentifiers & Categorical Fields:")
print(f"- Primary Identifier: LocationID (Range: {lookup_df['LocationID'].min()} to {lookup_df['LocationID'].max()})")
print(f"- Boroughs represented: {lookup_df['Borough'].unique().tolist()}")
print(f"- Service Zones: {lookup_df['service_zone'].unique().tolist()}")

print("\nSample Data:")
print(lookup_df.head(3).to_string(index=False))


# 2. Audit Spatial Geometries (GeoParquet or Shapefile)
print("\n" + "=" * 60)
print("\n--- 2. TAXI ZONES SPATIAL FILE (GeoParquet / Shapefile) ---")
try:
    # Try loading GeoParquet first
    zones_gdf = gpd.read_parquet("../data/spatial/taxi_zones.parquet")
    source_type = "GeoParquet"
except Exception:
    # Fallback to Shapefile bundle if parquet isn't present
    zones_gdf = gpd.read_file("../data/spatial/taxi_zones.shp")
    source_type = "Shapefile"

print(f"Source Format: {source_type}")
print(f"Total Polygons (Rows): {len(zones_gdf):,}")
print(f"Total Columns: {len(zones_gdf.columns)}")
print("\nColumns and Data Types:")
print(zones_gdf.dtypes)

print(f"\nCoordinate Reference System (CRS): {zones_gdf.crs}")

print("\nSample Data (excluding geometry coordinates for display):")
print(zones_gdf.drop(columns='geometry').head(3).to_string(index=False))

print("\n" + "=" * 60)
print("Spatial audit complete!")

SPATIAL DATA STRUCTURE & SCHEMA AUDIT

--- 1. TAXI ZONE LOOKUP TABLE (CSV) ---
Total Rows (Zones): 265
Total Columns: 4

Columns and Data Types:
LocationID      int64
Borough           str
Zone              str
service_zone      str
dtype: object

Identifiers & Categorical Fields:
- Primary Identifier: LocationID (Range: 1 to 265)
- Boroughs represented: ['EWR', 'Queens', 'Bronx', 'Manhattan', 'Staten Island', 'Brooklyn', 'Unknown', nan]
- Service Zones: ['EWR', 'Boro Zone', 'Yellow Zone', 'Airports', nan]

Sample Data:
 LocationID Borough                    Zone service_zone
          1     EWR          Newark Airport          EWR
          2  Queens             Jamaica Bay    Boro Zone
          3   Bronx Allerton/Pelham Gardens    Boro Zone


--- 2. TAXI ZONES SPATIAL FILE (GeoParquet / Shapefile) ---
Source Format: Shapefile
Total Polygons (Rows): 263
Total Columns: 7

Columns and Data Types:
OBJECTID         int32
Shape_Leng     float64
Shape_Area     float64
zone               st

In [11]:
import duckdb

# Connect to DuckDB in-memory database
con = duckdb.connect(database=':memory:')

print("=" * 70)
print("1. QUANTIFYING MISSING VALUES & IMPOSSIBLE VALUES (Including Micro-Trip Speed Artifacts)")
print("=" * 70)

quality_query = """
WITH base AS (
    SELECT 
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        passenger_count,
        trip_distance,
        fare_amount,
        PULocationID,
        DOLocationID,
        EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 60.0 AS duration_min,
        CASE 
            WHEN EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) > 0 
            THEN trip_distance / (EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 3600.0)
            ELSE NULL 
        END AS speed_mph
    FROM '../data/raw/yellow_tripdata_2025-*.parquet'
)
SELECT 
    COUNT(*) AS total_rows,
    
    -- Missing Values (NULL counts)
    SUM(CASE WHEN tpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS missing_pickup,
    SUM(CASE WHEN tpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS missing_dropoff,
    SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS missing_pulocation,
    SUM(CASE WHEN PULocationID = 264 THEN 1 ELSE 0 END) AS unknown_pulocation,
    SUM(CASE WHEN PULocationID = 265 THEN 1 ELSE 0 END) AS outside_nyc_pulocation,
    SUM(CASE WHEN DOLocationID IS NULL  THEN 1 ELSE 0 END) AS missing_dolocation,
    SUM(CASE WHEN DOLocationID = 265 THEN 1 ELSE 0 END) AS outside_nyc_dolocation,
    SUM(CASE WHEN DOLocationID = 264 THEN 1 ELSE 0 END) AS unknown_dolocation,
    SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) AS missing_distance,
    SUM(CASE WHEN fare_amount IS NULL THEN 1 ELSE 0 END) AS missing_fare,
    
    -- Impossible / Invalid Values (with conditional distance thresholds)
    SUM(CASE WHEN duration_min <= 0 OR duration_min > 240 OR duration_min IS NULL THEN 1 ELSE 0 END) AS invalid_duration,
    SUM(CASE WHEN trip_distance < 0 OR (DOLocationID != 265 AND trip_distance > 100) OR (DOLocationID = 265 AND trip_distance > 200) THEN 1 ELSE 0 END) AS invalid_distance,
    SUM(CASE WHEN trip_distance = 0 THEN 1 ELSE 0 END) AS zero_distance,
    SUM(CASE WHEN speed_mph IS NULL OR speed_mph > 80 THEN 1 ELSE 0 END) AS invalid_speed,
    
    -- Explicit Micro-Trip Speed Artifact Count (Duration < 1 min AND Speed > 70 mph)
    SUM(CASE WHEN duration_min > 0 AND duration_min < 1.0 AND speed_mph > 70 THEN 1 ELSE 0 END) AS micro_trips_speed_gt_70,
    
    SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) AS negative_fare,
    SUM(CASE WHEN passenger_count < 0 OR passenger_count > 9 OR passenger_count IS NULL THEN 1 ELSE 0 END) AS invalid_passenger_count
FROM base;
"""

df_quality = con.execute(quality_query).df()
con.close()

print(df_quality.T)

1. QUANTIFYING MISSING VALUES & IMPOSSIBLE VALUES (Including Micro-Trip Speed Artifacts)
                                  0
total_rows               48722602.0
missing_pickup                  0.0
missing_dropoff                 0.0
missing_pulocation              0.0
unknown_pulocation          80873.0
outside_nyc_pulocation      22170.0
missing_dolocation              0.0
outside_nyc_dolocation     219073.0
unknown_dolocation         105193.0
missing_distance                0.0
missing_fare                    0.0
invalid_duration           563024.0
invalid_distance             2429.0
zero_distance             1402958.0
invalid_speed              560004.0
micro_trips_speed_gt_70      8543.0
negative_fare             2848620.0
invalid_passenger_count  11611894.0


In [8]:
import duckdb

# Connect to DuckDB in-memory database
con = duckdb.connect(database=':memory:')

print("=" * 80)
print("EXTRACTING 10 SAMPLES PER ANOMALY CATEGORY (INCLUDING MICRO-TRIP ARTIFACTS)")
print("=" * 80)

# Helper function to query and display samples joined with taxi zones
def print_anomaly_samples(title, condition_sql):
    query = f"""
    WITH raw_data AS (
        SELECT 
            VendorID,
            tpep_pickup_datetime,
            tpep_dropoff_datetime,
            passenger_count,
            trip_distance,
            fare_amount,
            PULocationID,
            DOLocationID,
            EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 60.0 AS duration_min,
            CASE 
                WHEN EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) > 0 
                THEN trip_distance / (EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 3600.0)
                ELSE NULL 
            END AS speed_mph
        FROM '../data/raw/yellow_tripdata_2025-*.parquet'
        WHERE {condition_sql}
        LIMIT 10
    )
    SELECT 
        r.tpep_pickup_datetime,
        r.tpep_dropoff_datetime,
        ROUND(r.duration_min, 2) AS duration_min,
        r.trip_distance,
        r.fare_amount,
        ROUND(r.speed_mph, 2) AS speed_mph,
        r.passenger_count,
        r.PULocationID,
        pu.Borough AS pu_borough,
        pu.Zone AS pu_zone,
        r.DOLocationID,
        d.Borough AS do_borough,
        d.Zone AS do_zone
    FROM raw_data r
    LEFT JOIN '../data/spatial/taxi_zone_lookup.csv' pu ON r.PULocationID = pu.LocationID
    LEFT JOIN '../data/spatial/taxi_zone_lookup.csv' d ON r.DOLocationID = d.LocationID;
    """
    
    print(f"\n--- {title} ---")
    df = con.execute(query).df()
    print(df.to_string(index=False))

# 1. Invalid / Excessive Duration (> 4 hours or <= 0 minutes)
print_anomaly_samples(
    "1. Invalid or Excessive Duration Anomalies (> 4h or <= 0 min)",
    "EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) <= 0 OR EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) > 14400"
)

# 2. Extreme Distance Anomalies (Intra-city > 100mi or Out-of-city > 200mi)
print_anomaly_samples(
    "2. Extreme Distance Anomalies (Intra-city > 100mi or Out-of-city > 200mi)",
    "(DOLocationID != 265 AND trip_distance > 100) OR (DOLocationID = 265 AND trip_distance > 200)"
)

# 3. Zero-Distance Trips
print_anomaly_samples(
    "3. Zero-Distance Trips (d = 0 miles)",
    "trip_distance = 0"
)

# 4. Negative Fare Amounts
print_anomaly_samples(
    "4. Negative Fare Amounts (< $0)",
    "fare_amount < 0"
)

# 5. Invalid or Missing Passenger Counts (NULL, 0, or > 9)
print_anomaly_samples(
    "5. Invalid or Missing Passenger Counts (NULL, 0, or > 9)",
    "passenger_count < 0 OR passenger_count > 9 OR passenger_count IS NULL"
)

# 6. Location Anomalies (Zone 264: Unknown (Either PO or DO) , Zone 265: Outside NYC (Only PO))
print_anomaly_samples(
    "6. Location Anomalies (PULocationID or DOLocationID in 264, 265)",
    "PULocationID IN (264, 265) OR DOLocationID = 264"
)

# 7. Invalid Speed Anomalies (Average Speed > 80 mph)
print_anomaly_samples(
    "7. Invalid Speed Anomalies (Average Speed > 80 mph)",
    "EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) > 0 AND trip_distance > 0 AND (trip_distance / (EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 3600.0)) > 80"
)

# 8. Micro-Trip Speed Artifacts (Duration < 1 min AND Speed > 70 mph)
print_anomaly_samples(
    "8. Micro-Trip Speed Artifacts (Duration < 1 min AND Speed > 70 mph)",
    "EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) > 0 AND EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 60.0 < 1.0 AND trip_distance > 0 AND (trip_distance / (EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 3600.0)) > 70"
)

con.close()

EXTRACTING 10 SAMPLES PER ANOMALY CATEGORY (INCLUDING MICRO-TRIP ARTIFACTS)

--- 1. Invalid or Excessive Duration Anomalies (> 4h or <= 0 min) ---
tpep_pickup_datetime tpep_dropoff_datetime  duration_min  trip_distance  fare_amount  speed_mph  passenger_count  PULocationID pu_borough                       pu_zone  DOLocationID do_borough                       do_zone
 2025-01-01 01:26:00   2025-01-02 01:22:08       1436.13           7.06        37.30       0.29                1           170  Manhattan                   Murray Hill            25   Brooklyn                   Boerum Hill
 2025-01-01 00:27:08   2025-01-02 00:00:00       1412.87           5.72        25.40       0.24                1           137  Manhattan                      Kips Bay            33   Brooklyn              Brooklyn Heights
 2025-01-01 00:18:22   2025-01-02 00:00:00       1421.63           1.12        17.00       0.05                1            50  Manhattan                  Clinton West            48  M